# 05 LoRA Train on Dataset3, Eval on Dataset4

Fine-tunes LoRA on `dataset3` (diverse unanswerable types) with improved hyperparameters:
- LR=1e-4, cosine schedule, warmup=0.1, weight_decay=0.01
- 90/10 train/val split with eval monitoring + early stopping
- Plots training curves
- Evaluates on `dataset4` and saves predictions for the LLM judge notebook

In [ ]:
%pip install -q accelerate bitsandbytes datasets matplotlib peft safetensors torch tqdm transformers

In [ ]:
# Compute check
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(props.total_memory / (1024**3), 2))
else:
    print("Running on CPU runtime")

In [ ]:
from pathlib import Path
import json
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/abstention-data")

if not (ROOT / "data" / "dataset3.jsonl").exists() or not (ROOT / "src").exists():
    raise FileNotFoundError(
        f"Invalid ROOT: {ROOT}. Make sure repo is in Drive at /content/drive/MyDrive/abstention-data"
    )

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)

In [ ]:
from abstention_pipeline.config import DEFAULT_MODEL_NAME, DEFAULT_SYSTEM_PROMPT, resolve_dataset_path
from abstention_pipeline.training import train_lora
from abstention_pipeline.modeling import load_model_with_adapter, load_tokenizer
from abstention_pipeline.evaluation import run_benchmark, save_report, save_predictions
from abstention_pipeline.visualization import load_trainer_state, extract_training_curves, plot_training_curves

MODEL_NAME = DEFAULT_MODEL_NAME
QUANT_MODE = "4bit"
SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT
TRAIN_DATASET = "dataset3"
EVAL_DATASET = "dataset4"

# Improved LoRA + training params (combat over-abstention)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4          # gentler than D2's 2e-4
NUM_EPOCHS = 1.0
PER_DEVICE_BATCH_SIZE = 4     # A100 can handle it
GRAD_ACCUM = 8
MAX_LENGTH = 2048
WARMUP_RATIO = 0.1            # prevents early abstention bias
WEIGHT_DECAY = 0.01           # mild regularization
LR_SCHEDULER = "cosine"       # smoother decay
LOGGING_STEPS = 20
SAVE_STEPS = 200
EVAL_STEPS = 200
EARLY_STOPPING_PATIENCE = 3

MAX_NEW_TOKENS = 32
EVAL_BATCH_SIZE = 16
MAX_EXAMPLES = None
MAX_INPUT_TOKENS = 512

RUN_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset3"
TRAIN_DIR = RUN_DIR / "training"
EVAL_DIR = RUN_DIR / "eval"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print("Run dir:", RUN_DIR)

## Split Dataset3 for Train/Val (90/10)

In [ ]:
import random
from abstention_pipeline.data import load_jsonl

dataset3_path = resolve_dataset_path(ROOT, TRAIN_DATASET)
all_rows = load_jsonl(dataset3_path)
print(f"Total dataset3 rows: {len(all_rows)}")

# Shuffle with fixed seed for reproducibility
random.seed(42)
random.shuffle(all_rows)

val_size = 1000
train_rows = all_rows[val_size:]
val_rows = all_rows[:val_size]
print(f"Train: {len(train_rows)}, Val: {len(val_rows)}")

# Save splits as temporary JSONL files
train_split_path = RUN_DIR / "dataset3_train.jsonl"
val_split_path = RUN_DIR / "dataset3_val.jsonl"

for path, rows in [(train_split_path, train_rows), (val_split_path, val_rows)]:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved train split: {train_split_path}")
print(f"Saved val split: {val_split_path}")

## Train LoRA on Dataset3

In [ ]:
# Optional resume: if checkpoints exist in TRAIN_DIR, continue from latest.
checkpoints = sorted(
    [p for p in TRAIN_DIR.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
)
RESUME_FROM_CHECKPOINT = str(checkpoints[-1]) if checkpoints else None
print("Resume from:", RESUME_FROM_CHECKPOINT)

In [ ]:
result = train_lora(
    model_name=MODEL_NAME,
    dataset_path=train_split_path,
    output_dir=TRAIN_DIR,
    system_prompt=SYSTEM_PROMPT,
    quantization_mode=QUANT_MODE,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_length=MAX_LENGTH,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    # Improved training params
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER,
    eval_dataset_path=val_split_path,
    eval_steps=EVAL_STEPS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)
print("Final adapter:", result["adapter_dir"])
print("Output dir:", result["output_dir"])

## Training Curves

In [ ]:
state = load_trainer_state(TRAIN_DIR)
curves = extract_training_curves(state)
plot_training_curves(
    curves,
    title="Dataset3 Training Curves (LR=1e-4, cosine, warmup=0.1)",
    save_path=RUN_DIR / "training_curves.png",
)

## Eval on Dataset4

In [ ]:
# Restart-safe: if kernel restarts, this points to the adapter saved in TRAIN_DIR.
ADAPTER_PATH = str(TRAIN_DIR / "final_adapter")

model, tokenizer = load_model_with_adapter(
    model_name=MODEL_NAME,
    adapter_path=ADAPTER_PATH,
    quantization_mode=QUANT_MODE,
)

report = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    dataset_path=resolve_dataset_path(ROOT, EVAL_DATASET),
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=EVAL_BATCH_SIZE,
    max_examples=MAX_EXAMPLES,
    max_input_tokens=MAX_INPUT_TOKENS,
)

# Save full report (backward compat)
save_report(report, EVAL_DIR / "dataset4.json")
# Save predictions-only (for judge notebook)
save_predictions(report["predictions"], EVAL_DIR / "dataset4_predictions.json")

print(json.dumps(report["metrics"], indent=2))
print("\nSaved:", EVAL_DIR / "dataset4.json")
print("Saved:", EVAL_DIR / "dataset4_predictions.json")

In [ ]:
# Quick EM metrics summary
from abstention_pipeline.metrics import compute_metrics
metrics = compute_metrics(report["predictions"])
print(json.dumps(metrics, indent=2))